## Concept 1 — Tool Calling Agent

### What is it?
The simplest agent pattern. The LLM is bound to a set of tools and decides which one(s)
to call for a given query. Your code executes the tool and sends the result back.
No explicit reasoning trace — just: query → tool decision → execute → answer.

### Pipeline
```
Query
  → LLM decides: which tool + what args
  → your code executes the tool
  → ToolMessage returned to LLM
  → LLM forms final answer
```

### Limitation (what Concept 2 fixes)
No reasoning trace — you see the tool decision but not *why* the LLM made it.
→ Concept 2 (ReAct) adds an explicit Thought before every Action.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries, calculate, search_docs, get_weather
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm()
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — Bind Tools to LLM
`bind_tools` registers the tools with the LLM. The LLM reads each tool's docstring
to understand what it does and when to call it.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

# Ask a simple question — watch what the LLM returns
response = llm_with_tools.invoke('What is 144 divided by 12?')

print(f'Content:    "{response.content}"')
print(f'Tool calls: {response.tool_calls}')
# Notice: content is empty — LLM chose to call a tool instead of answering directly

### Step 2 — Execute the Tool Call
Your code reads the tool call, runs the function, and wraps the result in a `ToolMessage`.

In [ ]:
tool_map = {t.name: t for t in TOOLS}

def execute_tool_calls(response):
    tool_messages = []
    for call in response.tool_calls:
        result = tool_map[call['name']].invoke(call['args'])
        print(f'  Tool: {call["name"]}({call["args"]}) → {result}')
        tool_messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
    return tool_messages

# Test with our first question
messages = [HumanMessage(content='What is 144 divided by 12?')]
r1       = llm_with_tools.invoke(messages)
messages.append(r1)

tool_msgs = execute_tool_calls(r1)
messages.extend(tool_msgs)

final = llm_with_tools.invoke(messages)
print(f'\nFinal answer: {final.content}')

### Step 3 — Direct Answer (No Tool Needed)
When the LLM can answer from its training data, it skips tools entirely.

In [ ]:
direct_q = 'What programming language is LangChain written in?'
r = llm_with_tools.invoke(direct_q)
print(f'Tool calls: {r.tool_calls}')   # empty — answered directly
print(f'Direct answer: {r.content}')

### Step 4 — Parallel Tool Calls
The LLM can call multiple tools in a single response when the query needs several results.

In [ ]:
multi_q = 'Search docs for RAG and also calculate 25 multiplied by 4'
messages = [HumanMessage(content=multi_q)]
r = llm_with_tools.invoke(messages)
messages.append(r)

print(f'Tool calls made: {len(r.tool_calls)}')
for call in r.tool_calls:
    print(f'  {call["name"]}({call["args"]})')

tool_msgs = execute_tool_calls(r)
messages.extend(tool_msgs)
final = llm_with_tools.invoke(messages)
print(f'\nFinal: {final.content}')

### Full Function

In [ ]:
def tool_calling_agent(query: str) -> str:
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        messages.extend(execute_tool_calls(response))
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content

### Test — Standard Queries

In [ ]:
run_queries(tool_calling_agent, label='Concept 1 — Tool Calling Agent')